# 02 — Improvement arms (post-RCA)

Baseline pipeline & analysis: `01model_yolov26n.ipynb`.
RCA found: FN-dominated (sand), ambiguous-region GT, val→test domain shift.
This notebook tests the **cheap, high-ROI levers** from `methods_survey.md`:

1. `copy_paste` augmentation (Ghiasi CVPR21, +3.6 mask AP rare classes)
2. `mask_ratio=1` + `retina_masks=True` (boundary fidelity at high IoU)
3. TTA (`augment=True`)
4. Per-class conf threshold sweep (sand FN recovery)

Same recipe as nb01 winner: **AdamW lr=1e-3, gold, seed=42, batch=12, ≤100ep, patience=30, mask-fitness selection**.

In [1]:
# ============================================================
# SETUP — env, constants, mask-fitness patch (same as nb01)
# ============================================================
import os, json, time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils.metrics import SegmentMetrics
from ultralytics.data.utils import check_det_dataset

WORK   = Path.cwd()
RESULTS = WORK / "results"; RESULTS.mkdir(exist_ok=True)
DATA   = {"gold": "data/gold/data.yaml"}
IMGSZ, SEED   = 640, 42
BATCH, DEVICE, WORKERS, PATIENCE = 12, 0, 8, 30
WINNER_OPT, WINNER_LR = "AdamW", 1e-3

# thesis-metric alignment: mask fitness drives early-stop + best.pt
SegmentMetrics.fitness = property(lambda m: m.seg.fitness())
print("patched: SegmentMetrics.fitness -> mask")

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
for k, v in DATA.items():
    d = check_det_dataset(v)
    import glob
    print(k, {s: len(glob.glob(f"data/gold/{s}/images/*")) for s in ("train","valid","test")})


patched: SegmentMetrics.fitness -> mask
torch 2.11.0+cu130 | cuda True
gold {'train': 753, 'valid': 226, 'test': 105}


In [2]:
# ============================================================
# ARM gold_boost — nb01 winner recipe + survey levers
#   copy_paste=0.2  (rare-class AP lever; sand is the weak class)
#   mask_ratio=1    (full-res mask targets -> high-IoU AP)
# ============================================================
AUG = dict(hsv_h=0.005, hsv_s=0.25, hsv_v=0.20, degrees=0.0, translate=0.02,
           scale=0.10, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.0,
           mosaic=0.30, mixup=0.0, copy_paste=0.20, erasing=0.0)

t0 = time.time()
m = YOLO("yolo26n-seg.pt")          # same pretrained stock as baseline
r = m.train(data=DATA["gold"], task="segment", epochs=100, imgsz=IMGSZ, batch=BATCH,
            device=DEVICE, workers=WORKERS, seed=SEED, project="runs",
            name="arm_gold_boost", exist_ok=True,
            optimizer=WINNER_OPT, lr0=WINNER_LR, lrf=0.01,
            weight_decay=5e-4, momentum=0.937, amp=True, close_mosaic=8,
            patience=PATIENCE, cache=False, save=True, save_period=10,
            plots=True, val=True, deterministic=True,
            mask_ratio=1, **AUG)
run_dir = Path(m.trainer.save_dir)
print("run:", run_dir, f"{(time.time()-t0)/60:.1f} min")


SyntaxError: f-string: unmatched ')' (2682247052.py, line 21)

In [ ]:
# ============================================================
# EVAL — val + test, per-class, same metrics as nb01
# ============================================================
import pandas as pd
best = YOLO(str(run_dir / "weights" / "best.pt"))

def mm_row(tag, split, m):
    names = m.names
    return dict(run=tag, split=split, mask_mAP50=round(m.seg.map50,4),
                mask_mAP5095=round(m.seg.map,4),
                gravel_95=round(m.seg.maps[names.get("gravel",0)],4),
                sand_95=round(m.seg.maps[names.get("sand",1)],4),
                P=round(m.seg.mp,4), R=round(m.seg.mr,4))

rows = [mm_row("boost", s, best.val(data=DATA["gold"], split=s, imgsz=IMGSZ,
        batch=BATCH, device=DEVICE, retina_masks=True, plots=False))
        for s in ("val","test")]

prev = pd.read_csv(RESULTS / "p2_ablation.csv")   # p3-baseline / p2 / p2w
out  = pd.concat([prev, pd.DataFrame(rows)], ignore_index=True)
out.to_csv(RESULTS / "boost_ablation.csv", index=False)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(out.to_string(index=False))


In [ ]:
# ============================================================
# TTA — augment=True on val+test (no retrain)
# ============================================================
tta = [mm_row("boost+tta", s, best.val(data=DATA["gold"], split=s, imgsz=IMGSZ,
       batch=1, device=DEVICE, retina_masks=True, augment=True, plots=False))
       for s in ("val","test")]
for r_ in tta: print(r_)


In [ ]:
# ============================================================
# Per-class conf sweep — recover sand FN without wrecking P
# ============================================================
from ultralytics.utils.metrics import ConfusionMatrix  # noqa
import numpy as np

imgs = sorted(Path("data/gold/test/images").iterdir())
sweep = []
for cf in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    m_ = best.val(data=DATA["gold"], split="test", imgsz=IMGSZ, batch=BATCH,
                  device=DEVICE, conf=cf, retina_masks=True, plots=False)
    names = m_.names
    sweep.append(dict(conf=cf, P=round(m_.seg.mp,3), R=round(m_.seg.mr,3),
                      mAP50_95=round(m_.seg.map,4),
                      sand_95=round(m_.seg.maps[names.get("sand",1)],4)))
print(pd.DataFrame(sweep).to_string(index=False))


## Notes for paper

- `boost` arm = nb01 winner recipe + copy_paste(0.2) + mask_ratio(1) + retina_masks.
- Compare against `p3-baseline`/`p2`/`p2w` rows — same data, seed, recipe, metric.
- TTA and per-class thresholds are inference-time; reported separately from training arms.
- If sand conf-sweep recovers FN at acceptable P, cite FN-dominant failure profile (fpfn_test overlays, nb01 §6.4).